# Session 2: Prompt Engineering for Agentic Behaviors (Student Notebook)

## Learning Objectives

By the end of this session, you will be able to:

1. **Craft effective system prompts** that define agent personas, capabilities, and constraints
2. **Implement Chain-of-Thought (CoT) and ReAct prompting** patterns to improve reasoning quality
3. **Generate structured outputs** (JSON mode) for reliable downstream processing
4. **Manage multi-turn conversations** with context accumulation and summarization strategies
5. **Use LangChain prompt templates** and output parsers for reusable, composable prompt workflows

In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# Imports and Setup
import openai
import json
import os

# Ensure your OPENAI_API_KEY is set in your environment variables
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

client = openai.OpenAI()
print("Setup complete. OpenAI client initialized.")

---
## Demos

Follow along with these demos. Run each cell and observe the outputs carefully.

### Demo 1: Zero-Shot vs. Few-Shot Prompting

**Zero-shot** prompting asks the model to perform a task with no examples.  
**Few-shot** prompting provides a handful of examples so the model can learn the expected format and behavior.

We will compare both approaches on a **client feedback classification** task — categorizing engagement survey responses as positive, negative, or neutral.

In [ ]:
# Demo 1: Zero-Shot vs. Few-Shot Prompting

# Zero-shot: no examples provided
zero_shot_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Classify the sentiment of this client engagement feedback as positive, negative, or neutral: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}]
)
print("Zero-shot:", zero_shot_response.choices[0].message.content)

print("\n" + "="*60 + "\n")

# Few-shot: provide labeled examples from past McKinsey engagement surveys
few_shot_messages = [
    {"role": "system", "content": "You are a McKinsey client feedback classifier. Respond with only: POSITIVE, NEGATIVE, or NEUTRAL."},
    {"role": "user", "content": "Feedback: 'The engagement exceeded our expectations — the team was exceptional and the insights transformed our strategy.'"},
    {"role": "assistant", "content": "POSITIVE"},
    {"role": "user", "content": "Feedback: 'Disappointed with the engagement. Deliverables were late and the analysis did not address our core issues.'"},
    {"role": "assistant", "content": "NEGATIVE"},
    {"role": "user", "content": "Feedback: 'The McKinsey team delivered solid analysis but the final recommendations lacked specificity for our market.'"}
]
few_shot_response = client.chat.completions.create(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), messages=few_shot_messages)
print("Few-shot:", few_shot_response.choices[0].message.content)

**Observation:** Notice how the zero-shot response may include extra explanation, while the few-shot response follows the exact format demonstrated in the examples.

### Demo 2: Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting encourages the model to show its reasoning process step by step, which improves accuracy on math, logic, and multi-step problems.

This mirrors McKinsey's structured problem-solving — breaking a problem into steps before jumping to the answer. CoT is especially valuable for market sizing and financial analysis.

In [ ]:
# Demo 2: Chain-of-Thought Prompting

problem = "A McKinsey client has 1,200 retail stores. They plan to close 20% of underperforming locations and increase revenue per remaining store by 15%. If current average revenue per store is $2M, what will total revenue be after the restructuring?"

# Without CoT
response_no_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": problem}]
)
print("Without CoT:")
print(response_no_cot.choices[0].message.content)

print("\n" + "="*60 + "\n")

# With explicit CoT — mirroring McKinsey structured problem-solving
cot_prompt = f"{problem}\n\nLet's solve this step by step:\n1. First, calculate how many stores will be closed\n2. Then, find the number of remaining stores\n3. Calculate the new revenue per store after the 15% increase\n4. Compute total projected revenue"
response_cot = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": cot_prompt}]
)
print("With CoT:")
print(response_cot.choices[0].message.content)

**Observation:** The CoT version explicitly walks through each calculation, making it easier to verify correctness and debug any errors in reasoning.

### Demo 3: Role-Based Prompting for Agentic Personas

By changing the system prompt, we can create entirely different McKinsey practice-area personas that approach the same client question from different angles. This is foundational for building specialized agents in multi-agent consulting systems.

In [ ]:
# Demo 3: Role-Based Prompting for Agentic Personas

personas = {
    "McKinsey Growth Strategy Lead": "You are a McKinsey partner in Growth & Innovation. You approach every question by identifying growth levers, market opportunities, and revenue acceleration strategies. Use frameworks like the Three Horizons of Growth.",
    "McKinsey Risk & Compliance Expert": "You are a McKinsey senior expert in Risk & Resilience. You evaluate everything through the lens of regulatory risk, compliance requirements, operational resilience, and enterprise risk management.",
    "McKinsey Organization & People Lead": "You are a McKinsey partner in People & Organizational Performance. You think about talent strategy, organizational design, change management, and leadership development. Focus on the human side of transformation."
}
question = "Our banking client is considering launching a digital-only subsidiary to compete with fintechs."

for role, system_prompt in personas.items():
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "200"))
    )
    print(f"\n{'='*50}\n{role}:\n{response.choices[0].message.content}")

**Observation:** Each persona focuses on different aspects of the same question -- data metrics, security risks, or user/business impact. This demonstrates how system prompts shape agent behavior.

### Demo 4: Structured Output Generation (JSON Mode)

When building agentic systems for McKinsey workflows, we often need the LLM output to be machine-readable — for example, extracting structured client data from meeting notes or parsing engagement details from unstructured text. JSON mode ensures the response is valid JSON, making downstream processing reliable.

In [ ]:
# Demo 4: Structured Output Generation (JSON Mode)

text = "Sarah Chen is the CFO of Meridian Healthcare, a $3.2B revenue hospital network based in Chicago. She has 18 years of experience in healthcare finance and previously led M&A at Deloitte. Her priorities for the McKinsey engagement are cost optimization, revenue cycle management, and evaluating two potential acquisition targets."

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "Extract the client executive's information and return it as a JSON object with keys: name, title, company, company_revenue, location, experience_years, previous_employer, engagement_priorities (as a list)."},
        {"role": "user", "content": text}
    ],
    response_format={"type": "json_object"}
)

parsed = json.loads(response.choices[0].message.content)
print("Parsed JSON output:")
print(json.dumps(parsed, indent=2))

# Demonstrate that we can access individual fields programmatically
print(f"\nClient Executive: {parsed.get('name')}, {parsed.get('title')}")
print(f"Engagement Priorities: {', '.join(parsed.get('engagement_priorities', []))}")

**Observation:** With `response_format={"type": "json_object"}`, the output is guaranteed to be valid JSON, which we can reliably parse and use in code.

### Demo 5: Multi-Turn Conversation Management

Consulting agents need to maintain context across multiple exchanges — remembering the client industry, engagement scope, and prior recommendations. This demo shows how to build a conversation manager that accumulates context, mirroring how a McKinsey engagement progresses through multiple discussions.

In [ ]:
# Demo 5: Multi-Turn Conversation Management

class ConversationManager:
    def __init__(self, system_prompt, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")):
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]
        self.client = openai.OpenAI()
    
    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
        )
        assistant_msg = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_msg})
        return assistant_msg
    
    def get_history_length(self):
        return len(self.messages)

# Demo: Multi-turn McKinsey engagement planning conversation
conv = ConversationManager("You are a McKinsey engagement planning assistant. Help consultants scope and plan client engagements. Be structured, use MECE principles, and provide actionable next steps.")

print("User: We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%.")
print("Assistant:", conv.send("We just won a new engagement with a mid-size insurance company. They want to reduce claims processing costs by 30%."))

print("\n" + "-"*50)
print("\nUser: What data should we request from the client in the first week?")
print("Assistant:", conv.send("What data should we request from the client in the first week?"))

print("\n" + "-"*50)
print("\nUser: Can you summarize our engagement scope so far?")
print("Assistant:", conv.send("Can you summarize our engagement scope so far?"))

print(f"\nConversation history: {conv.get_history_length()} messages")

**Observation:** The assistant remembers context from earlier turns -- it knows we are discussing Japan without being reminded. The conversation history grows with each exchange.

### Demo 6: Structured Outputs with LangChain Prompt Templates

LangChain provides a higher-level abstraction for prompt engineering. Its `ChatPromptTemplate` supports parameterized prompts, and `OutputParser` classes enforce structured output formats (JSON, lists, etc.) — reducing boilerplate and improving reliability.

In [ ]:
# Demo 6: Structured Outputs with LangChain Prompt Templates

# Install if needed: pip install langchain langchain-openai langchain-community
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# Step 1: Initialize the LangChain LLM wrapper
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# --- Part A: ChatPromptTemplate with variables ---
print("PART A: LangChain ChatPromptTemplate for McKinsey Analysis")
print("=" * 60)

# Reusable template for industry analysis across different client engagements
analysis_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey {practice_area} expert. Provide structured, partner-ready analysis."),
    ("user", "Analyze the following for our client engagement: {topic}\n\nProvide your analysis in exactly 3 bullet points with strategic implications.")
])

prompt_value = analysis_template.invoke({"practice_area": "Operations", "topic": "supply chain resilience in the semiconductor industry"})
response = llm.invoke(prompt_value)
print(f"Practice: Operations | Topic: semiconductor supply chain")
print(response.content)

print()
prompt_value2 = analysis_template.invoke({"practice_area": "Strategy & Corporate Finance", "topic": "consolidation trends in European banking"})
response2 = llm.invoke(prompt_value2)
print(f"Practice: Strategy | Topic: European banking consolidation")
print(response2.content)

# --- Part B: Pydantic-based Structured Output for McKinsey deliverables ---
print("\n" + "=" * 60)
print("PART B: Pydantic Structured Output — McKinsey Engagement Summary")
print("=" * 60)

class EngagementSummary(BaseModel):
    executive_summary: str = Field(description="A 1-2 sentence executive summary suitable for a CEO briefing")
    key_findings: list[str] = Field(description="A list of 3 key findings")
    strategic_priority: str = Field(description="The single most important strategic priority: growth, cost_optimization, digital_transformation, or M&A")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")

parser = JsonOutputParser(pydantic_object=EngagementSummary)
format_instructions = parser.get_format_instructions()
print(f"Format instructions (auto-generated):\n{format_instructions}\n")

structured_template = ChatPromptTemplate.from_messages([
    ("system", "You are a McKinsey engagement manager preparing a structured deliverable."),
    ("user", "Analyze the strategic landscape for: {topic}\n\n{format_instructions}")
])

prompt = structured_template.invoke({
    "topic": "a mid-size US retailer considering an omnichannel transformation",
    "format_instructions": format_instructions
})

response = llm.invoke(prompt)
parsed_output = parser.parse(response.content)

print("Parsed structured output:")
for key, value in parsed_output.items():
    print(f"  {key}: {value}")

# --- Part C: LangChain Chain (Prompt | LLM | Parser) ---
print("\n" + "=" * 60)
print("PART C: LangChain Chain — Prompt | LLM | Parser")
print("=" * 60)

chain = structured_template | llm | parser

result = chain.invoke({
    "topic": "a healthcare provider evaluating AI-powered diagnostics for radiology",
    "format_instructions": format_instructions
})

print("Chain output (directly parsed):")
print(json.dumps(result, indent=2))

---
## Tasks

Now it is your turn! Complete the following tasks by filling in the `TODO` sections. Use the demo code above as reference.

### Task 1: Design Effective System Prompts for a McKinsey Industry Research Agent

Create a system prompt for a McKinsey "Industry Research Agent" and test it with a research question about the electric vehicle battery market.

In [ ]:
# Task 1: Design Effective System Prompts for a McKinsey Industry Research Agent

# TODO: Create a system prompt for a McKinsey "Industry Research Agent" that:
# 1. Defines the agent's role as a McKinsey industry research specialist
# 2. Specifies how it should approach research questions (MECE breakdown, data-driven reasoning)
# 3. Defines output format (structured with sections: Executive Summary, Key Findings, Strategic Implications, Confidence Level, Data Gaps)
# 4. Includes constraints (acknowledge uncertainty, use McKinsey frameworks, flag assumptions)
#
# Then test it with a research question about the EV battery market.
#
# Hint: Good system prompts are 100-300 words
# Hint: Include specific output formatting instructions
# Hint: Add behavioral guidelines (e.g., "If uncertain, explicitly state your confidence level")
# Hint: Reference McKinsey frameworks (Porter's Five Forces, value chain analysis, etc.)

research_agent_prompt = ""  # YOUR CODE HERE

def ask_research_agent(question):
    # YOUR CODE HERE
    pass

# Test it
# ask_research_agent("What are the key growth drivers and competitive dynamics in the global electric vehicle battery market?")

### Task 2: Implement ReAct-Style Prompting (Reason + Act)

Implement a ReAct-style prompt where the model interleaves reasoning (Thought) with actions (Action) and observations (Observation) — applied to a McKinsey due diligence scenario.

In [ ]:
# Task 2: Implement ReAct-Style Prompting (Reason + Act)

# TODO: Implement a ReAct-style prompt that follows the pattern:
# Thought: [hypothesis or reasoning about what data you need next]
# Action: [what action to take]
# Observation: [result of the action]
# ... (repeat)
# Final Answer: [structured recommendation]
#
# Apply this to a McKinsey due diligence scenario.
# The agent should reason through market data, financials, and competitor analysis.
#
# Hint: Define available "actions" in the system prompt:
#   - MarketData[query]: Look up market size, growth rates, and industry data
#   - FinancialAnalysis[expression]: Perform financial calculations (margins, multiples, CAGR)
#   - CompetitorLookup[company]: Research competitor positioning and market share
# Hint: Structure the system prompt with the ReAct format template
# Hint: The LLM simulates the actions since we don't have real tools yet

def react_agent(question):
    # YOUR CODE HERE
    pass

# Test it
# react_agent("Is a $200M acquisition of a mid-size European logistics company a good deal for our private equity client?")

### Task 3: Create a Reusable McKinsey Prompt Template Library

Build a `PromptTemplate` class with variable placeholders, validation, and a library of pre-built McKinsey consulting templates.

In [ ]:
# Task 3: Create a Reusable McKinsey Prompt Template Library

# TODO: Build a PromptTemplate class that:
# 1. Stores template strings with {variable} placeholders
# 2. Has a .format() method to fill in variables
# 3. Has a .validate() method to check all variables are provided
# 4. Includes at least 3 pre-built McKinsey consulting templates:
#    - market_sizing_template: TAM estimation for a product category in a geography
#    - executive_briefing_template: CEO briefing on a topic for an industry client
#    - competitive_analysis_template: Competitive landscape for a company in a sector
#
# Hint: Use Python string .format() or f-strings
# Hint: Use re.findall(r'\{(\w+)\}', template) to find placeholders
# Hint: Raise an error if required variables are missing

import re

class PromptTemplate:
    def __init__(self, name, template, description=""):
        # YOUR CODE HERE
        pass
    
    def format(self, **kwargs):
        # YOUR CODE HERE
        pass
    
    def validate(self, **kwargs):
        # YOUR CODE HERE
        pass

# Create McKinsey consulting template library
# market_sizing_template = PromptTemplate(...)
# executive_briefing_template = PromptTemplate(...)
# competitive_analysis_template = PromptTemplate(...)

### Task 4: Build a McKinsey Analyst Agent with Consulting Tools

Build a complete agentic loop where the LLM acts as a McKinsey analyst, deciding which consulting tool to call, receiving simulated results, and iterating until it has a final recommendation.

In [ ]:
# Task 4: Build a McKinsey Analyst Agent with Consulting Tools

# TODO: Build an agent that:
# 1. Has a system prompt describing available McKinsey consulting tools:
#    - market_database: Look up market size, growth rates, and industry trends
#    - financial_model: Run financial calculations (DCF, multiples, CAGR, margin analysis)
#    - benchmarking: Compare client KPIs against industry benchmarks
# 2. Receives a complex consulting question requiring multiple analytical steps
# 3. Uses structured output (JSON) to specify which tool to call at each step:
#    {"thought": "analytical reasoning", "tool": "tool_name", "input": "tool_input"}
# 4. Processes the "tool results" and generates a final recommendation
#
# Hint: Define tools as a list of dicts with name, description, parameters
# Hint: Ask the LLM to respond with JSON: {"thought": "...", "tool": "name", "input": "..."}
# Hint: Use "final_answer" as the tool name when the agent is ready to give its recommendation
# Hint: Simulate tool responses and feed them back into the conversation
# Hint: Add a max-step limit to prevent infinite loops

tools = [
    {"name": "market_database", "description": "Look up market size, growth rates, and industry trends", "parameters": "query (string)"},
    {"name": "financial_model", "description": "Run financial calculations: DCF, multiples, CAGR, margin analysis", "parameters": "expression (string)"},
    {"name": "benchmarking", "description": "Compare client KPIs against industry benchmarks", "parameters": "metric and industry (string)"}
]

class ToolAgent:
    def __init__(self):
        # YOUR CODE HERE
        pass
    
    def simulate_tool(self, tool_name, tool_input):
        # YOUR CODE HERE
        pass
    
    def process(self, question):
        # YOUR CODE HERE
        pass

# Test: "Should our PE client acquire a specialty chemicals company with $80M revenue and 22% EBITDA margins?"

### Task 5: Prompt Chaining for M&A Due Diligence Pipeline

**Goal:** Implement a 3-stage prompt chain where the output of each LLM call feeds directly into the next — simulating a McKinsey M&A due diligence pipeline: **extract metrics -> strategic fit -> recommendation**.

This pattern decomposes complex analysis into modular, verifiable stages — each with a focused prompt and structured JSON output for reliable handoff.

In [ ]:
# Task 5: Prompt Chaining for M&A Due Diligence Pipeline

# TODO: Build a 3-stage prompt chain where each stage's output feeds into the next:
#   Stage 1: Extract financial/operational metrics from a company description -> JSON
#   Stage 2: Assess strategic fit using Stage 1's metrics -> JSON
#   Stage 3: Generate investment recommendation using Stage 1 + Stage 2 output -> JSON
#
# Hint: Use response_format={"type": "json_object"} at each stage for reliable handoff
# Hint: json.loads() the response, then pass it into the next stage's prompt with json.dumps()
# Hint: Each stage should have a focused system prompt (analyst, strategist, partner)
# Hint: The key pattern is: output_1 = stage1(input) -> output_2 = stage2(output_1) -> ...

def ma_due_diligence_pipeline(company_description):
    """3-stage M&A due diligence pipeline using prompt chaining."""
    client = openai.OpenAI()
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")

    # Stage 1: Extract key financial and operational metrics
    # YOUR CODE HERE

    # Stage 2: Assess strategic fit (use Stage 1 output)
    # YOUR CODE HERE

    # Stage 3: Generate investment recommendation (use Stage 1 + Stage 2 output)
    # YOUR CODE HERE

    pass

# Test the pipeline
# ma_due_diligence_pipeline(
#     "TechFlow Solutions is a mid-market industrial IoT company headquartered in Munich, Germany. "
#     "Founded in 2015, it generates $120M in annual revenue with 28% EBITDA margins and 18% YoY growth. "
#     "The company employs 450 people and specializes in predictive maintenance sensors and cloud-based "
#     "analytics platforms for manufacturing clients."
# )

### Task 6: Self-Consistency Voting for Market Sizing

**Goal:** Run the same market-sizing prompt multiple times with temperature > 0, then aggregate the results to produce a more robust estimate — applying the self-consistency technique.

This mirrors how McKinsey teams **triangulate estimates from multiple analysts**. Higher temperature introduces diversity in reasoning paths; the median provides a more robust estimate than any single response.

In [ ]:
# Task 6: Self-Consistency Voting for Market Sizing

# TODO: Implement self-consistency by running the same market-sizing prompt N times
# and aggregating the results for a more robust estimate.
#   1. Create a prompt asking for TAM estimation in JSON format (tam_billion_usd, methodology, etc.)
#   2. Run it N times (e.g., 5) with temperature > 0 for diverse reasoning paths
#   3. Collect all tam_billion_usd values
#   4. Compute mean, median, std deviation, and range
#   5. Report the median as the recommended estimate
#
# Hint: Use temperature=0.7 or 0.8 to introduce diversity across samples
# Hint: import statistics; statistics.mean(), statistics.median(), statistics.stdev()
# Hint: The key insight is that median is more robust to outliers than mean
# Hint: Use response_format={"type": "json_object"} so you can parse tam_billion_usd reliably

import statistics

def self_consistent_market_sizing(product, geography, num_samples=5, temperature=0.7):
    """Run market-sizing prompt N times and aggregate for a robust estimate."""
    client = openai.OpenAI()
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")

    estimates = []
    for i in range(num_samples):
        # YOUR CODE HERE — call the LLM with the market-sizing prompt
        # Parse the JSON response and extract tam_billion_usd
        pass

    # YOUR CODE HERE — compute mean, median, stdev, range
    # Print aggregated results

    pass

# Test
# self_consistent_market_sizing("AI-powered clinical decision support systems", "North America")

### Task 7: Multi-Persona Debate for Strategic Decisions

**Goal:** Implement a multi-persona debate where a Bull analyst and Bear analyst argue opposite sides of an investment thesis, then a Moderator synthesizes a balanced recommendation.

This mirrors McKinsey's **"red team / blue team"** practice for stress-testing strategic recommendations. Each persona has distinct system prompts that shape their analytical lens.

In [ ]:
# Task 7: Multi-Persona Debate for Strategic Decisions

# TODO: Implement a multi-persona debate:
#   1. Define a Bull analyst system prompt (growth opportunities, reasons to invest)
#   2. Define a Bear analyst system prompt (risks, reasons for caution)
#   3. Define a Moderator system prompt (synthesize, recommend)
#   4. Run N rounds: Bull argues -> Bear responds (each sees prior debate history)
#   5. Moderator synthesizes the full debate into a recommendation
#
# Hint: Maintain a debate_history list that accumulates all arguments
# Hint: Feed the debate_history to each persona so they can respond to previous points
# Hint: The moderator receives the full transcript at the end
# Hint: Use distinct system prompts to create genuinely different analytical perspectives

def strategic_debate(thesis_statement, num_rounds=2):
    """Bull vs. Bear debate on an investment thesis, with moderator synthesis."""
    client = openai.OpenAI()
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")

    bull_system = ""   # YOUR CODE HERE — bull analyst system prompt
    bear_system = ""   # YOUR CODE HERE — bear analyst system prompt
    moderator_system = ""  # YOUR CODE HERE — moderator system prompt

    debate_history = []

    for round_num in range(1, num_rounds + 1):
        # Bull argues
        # YOUR CODE HERE

        # Bear responds
        # YOUR CODE HERE
        pass

    # Moderator synthesizes
    # YOUR CODE HERE

    pass

# Test
# strategic_debate(
#     "A global pharmaceutical company should acquire a mid-size biotech firm "
#     "specializing in mRNA therapeutics for $2.5B, representing a 40% premium to market cap."
# )

### Task 8: Guardrails & Output Validation for Client Deliverables

**Goal:** Build an output validation system that checks LLM-generated content against a McKinsey quality checklist and automatically retries if the output fails validation.

This pattern uses an **LLM-as-judge** to evaluate outputs against specific criteria — mirroring McKinsey's quality review process. Auto-retry with feedback loops allow the system to self-correct without manual intervention.

In [ ]:
# Task 8: Guardrails & Output Validation for Client Deliverables

# TODO: Build an output validation system with auto-retry:
#   1. Define a quality checklist (5 criteria for McKinsey deliverables)
#   2. Generate a strategic briefing on the given topic
#   3. Use a second LLM call (LLM-as-judge) to validate against the checklist -> JSON
#   4. If validation fails, append feedback to the prompt and retry (up to max_retries)
#   5. Return the deliverable once it passes, or the best-effort after max retries
#
# Hint: The quality checklist might include: executive summary, data points,
#       actionable recommendations, MECE structure, next steps
# Hint: Use response_format={"type": "json_object"} for the validation response
# Hint: The validation JSON should include: passed (bool), score, failures (list), feedback (str)
# Hint: Append the feedback to generation_prompt for the retry — the model learns from its mistakes

def validated_deliverable(topic, max_retries=3):
    """Generate a McKinsey deliverable with automatic quality validation and retry."""
    client = openai.OpenAI()
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")

    quality_checklist = [
        # YOUR CODE HERE — define 5 quality criteria
    ]

    generation_prompt = ""  # YOUR CODE HERE — prompt to generate a strategic briefing

    for attempt in range(1, max_retries + 1):
        # Step 1: Generate the deliverable
        # YOUR CODE HERE

        # Step 2: Validate against checklist using LLM-as-judge
        # YOUR CODE HERE

        # Step 3: If passed, return; otherwise append feedback and retry
        # YOUR CODE HERE
        pass

    pass

# Test
# validated_deliverable("The strategic implications of generative AI adoption for a top-10 US bank")

---
## Summary

### Key Takeaways

1. **Zero-shot vs. Few-shot**: Few-shot prompting provides examples that guide the model toward consistent format and behavior. Use it when you need predictable, structured responses.

2. **Chain-of-Thought (CoT)**: Explicitly asking for step-by-step reasoning improves accuracy on complex tasks. CoT is especially valuable for math, logic, and multi-step problems.

3. **Role-Based Prompting**: System prompts are the primary mechanism for defining agent personas. A well-crafted system prompt shapes the entire behavior profile of an agent.

4. **Structured Outputs**: JSON mode ensures machine-readable responses, which is critical for agentic workflows where outputs feed into downstream code or other agents.

5. **Multi-Turn Management**: Maintaining conversation history enables context-aware interactions, but requires careful management of the context window as conversations grow.

### Looking Ahead

These prompt engineering techniques form the foundation for building agentic systems. In the next session, we will explore how to evaluate and measure the quality of agent outputs systematically.